# We are comparing the baseline methods with our multi-agent method

## First, we load the questions that GPT-5-nano never got right in all 4 attempts

In [1]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)
from data_processing.data_analysis import select_problem_sample_for_model 
gpt5_nano="GPT-5-nano (high)"
df = select_problem_sample_for_model(gpt5_nano)
df

,Unnamed: 0,source,problem,competition,unique_problem_label,correct,parsed_answer,gold_answer,output_cost_per_tokens,problem_idx,cost,output_tokens,input_tokens,answer,user_message,idx_answer,model_config,model_name,input_cost_per_tokens,ten_percentile_group
5836,3928,NaN,Let $N$ denote the numbers of ordered triples ...,MathArena/aime_2025_outputs,MathArena/aime_2025: 15,False,147,735,0.4,15,0.024355,60875.0,107.0,\boxed{147},"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,1
15668,3920,NaN,A plane $\mathcal{P}$ intersects a rectangular...,MathArena/hmmt_feb_2025_outputs,MathArena/hmmt_feb_2025: 30,False,sqrt(110),\sqrt{\frac{95}{24}},0.4,30,0.017099,42737.0,79.0,Let the six edges of the hexagonal cross-secti...,"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,2
20160,2292,NaN,Consider all positive multiples of $77$ less t...,MathArena/cmimc_2025_outputs,MathArena/cmimc_2025: 5,False,25,194832,0.4,5,0.002802,6995.0,76.0,### Final answer\n\nReasoning:\n- The positive...,"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,4
24100,2552,NaN,Let $A B C D E F$ be a convex cyclic hexagon. ...,MathArena/brumo_2025_outputs,MathArena/brumo_2025: 25,False,R,\frac{1+\sqrt{31}}{2},0.4,25,0.011091,27716.0,84.0,- Let the circle’s circumradius be R. For the ...,"Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,6
24116,2568,NaN,"$4$ bears - Aruno, Bruno, Cruno and Druno - ar...",MathArena/brumo_2025_outputs,MathArena/brumo_2025: 21,False,250,160,0.4,21,0.006556,16368.0,181.0,"Step 1: Let a, b, c, d be the four positive in...","Please reason step by step, and put your final...",0,openai/gpt-5-nano,GPT-5-nano (high),0.05,8


We have 5 problems that GPT-5 nano couldn't solve in various ranges of difficulty. We'll go from the easiest to hardest.

## Problem 1 - Difficulty level 8

In [2]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==8].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                             MathArena/brumo_2025: 21
answer                  Step 1: Let a, b, c, d be the four positive in...
gold_answer                                                           160
ten_percentile_group                                                    8
problem                 $4$ bears - Aruno, Bruno, Cruno and Druno - ar...
Name: 24116, dtype: object

In [3]:
import textwrap

first_problem_description = df_pruned["problem"]

first_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=first_problem_description, width=80))
print("answer", first_problem_gold_answer)

$4$ bears - Aruno, Bruno, Cruno and Druno - are each given a card with a
positive integer and are told that the sum of their $4$ numbers is $17$. They
cannot show each other their cards, but discuss a series of observations in the
following order:  Aruno: "I think it is possible that the other three bears all
have the same card." Bruno: "At first, I thought it was possible for the other
three bears to have the same card. Now I know it is impossible for them to have
the same card." Cruno: "I think it is still possible that the other three bears
have the same card." Druno: "I now know what card everyone has." What is the
product of their four card values?
answer 160


### Reflexion attempt

In [4]:
from multi_agent.multi_agent import Problem
from prompt_template import Reflexion_Solver, Reflector
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=first_problem_description,
    answer=first_problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> $4$ bears - Aruno, Bruno, Cruno and Druno - are each given
a card with a positive integer and are told that the sum of their $4$ numbers is
$17$. They cannot show each other their cards, but discuss a series of
observations in the following order:  Aruno: "I think it is possible that the
other three bears all have the same card." Bruno: "At first, I thought it was
possible for the other three bears to have the same card. Now I know it is
impossible for them to have the same card." Cruno: "I think it is still possible
that the other three bears have the same card." Druno: "I now know what card
everyone has." What is the product of their four card values?.<<  "When you are
done, you should submidt your answer as: ANSWER: <your answer>.  No latex
formatting, just the raw number/numbers or strings at the very end.  Before you
start sharing your toughts, give a little summary of the conversation so

In [6]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [9]:
from reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
Let a, b, c, d be the four positive integers on the cards, with a + b + c + d = 17.

1) Aruno: “I think it is possible that the other three bears have the same card.”
- This means there exists x > 0 such that b = c = d = x and a + 3x = 17.
- Hence a = 17 − 3x > 0, so x can be 1,2,3,4,5, giving a ∈ {14,11,8,5,2}.
- Equivalently, Aruno’s a must be in A = {2, 5, 8, 11, 14}.

2) Bruno: “At first, I thought it was possible for the other three bears to have the same card. Now I know it is impossible for them to have the same card.”
- Initially (before Aruno spoke) Bruno would think a=c=d could happen iff there exists x > 0 with b + 3x = 17, i.e., 17 − b is divisible by 3.
  This requires b ∈ {2, 5, 8, 11, 14}.
- After hearing Aruno’s statement, Bruno must conclude that no a ∈ A can equal x = (17 − b)/3. So (17 − b)/3 ∉ A.
- Checking b ∈ {2,5,8,11,14}:
  - b = 2 → x = 5 ∈ A (possible)  
  - b = 5 → x = 4 ∉ A (impossible now)
  - b = 8 → x = 3 ∉ A (impossible now)
  - b = 11

The Reflexion agent solved this problem in 1 attempt

### Tree of thought attempt

In [6]:
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=first_problem_description, answer=first_problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

path d:\NLP-group-15
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ("- Step 1: From Aruno's statement, a is such that 17 − a can be written as 3t with t a positive integer. So 17 − a ≡ 0 (mod 3) and 17 − a > 0, giving a ∈ {2, 5, 8, 11, 14}.\n\n- Step 2: Bruno says: initially it was possible for A, C, D to be equal, but after hearing Aruno, it’s now impossible. If A, C, D were equal, then a = c = d = t and 17 − b = 3t. For such a triple to be consistent with Aruno’s constraint a ∈ {2, 5, 8, 11, 14}, t must be in {2, 5, 8, 11, 14}. But 17 − b must also be 3t, so 3t ≤ 16, giving t ∈ {2, 5} and hence b ∈ {11, 2}. Since Bruno now says it’s impossible, b ≠ 2, 11. Together with the initial requirement that 17 − b is divisible by 3 (for the initial possibility), we get b ∈ {5, 8, 14}.\n\n- Step 3: Cruno says it’s still possible that A, B, D are equal. If a = b = d = t, then 17 − c 

The tree of thought starts out with a wrong answer but eventually gets to the right answer after branching a few times.

### Solver-Rejector method - The proposed method

In [6]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

first_problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=first_problem_description, 
  answer=first_problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem1" ,n_steps=10, problem=first_problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

Our method didn't rank the right answer as the highest in this case.

## Problem 2 - Difficulty Level 6

In [11]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==6].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                             MathArena/brumo_2025: 25
answer                  - Let the circle’s circumradius be R. For the ...
gold_answer                                         \frac{1+\sqrt{31}}{2}
ten_percentile_group                                                    6
problem                 Let $A B C D E F$ be a convex cyclic hexagon. ...
Name: 24100, dtype: object

In [12]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

Let $A B C D E F$ be a convex cyclic hexagon. Suppose that $A B=D E=\sqrt{5}, B
C=E F=3$, and $C D=F A=\sqrt{20}$. Compute the circumradius of $A B C D E F$.
answer \frac{1+\sqrt{31}}{2}


### Reflexion Attempt

In [13]:
from multi_agent.multi_agent import Problem
from prompt_template import Reflexion_Solver, Reflector
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> Let $A B C D E F$ be a convex cyclic hexagon. Suppose that
$A B=D E=\sqrt{5}, B C=E F=3$, and $C D=F A=\sqrt{20}$. Compute the circumradius
of $A B C D E F$..<<  "When you are done, you should submidt your answer as:
ANSWER: <your answer>.  No latex formatting, just the raw number/numbers or
strings at the very end.  Before you start sharing your toughts, give a little
summary of the conversation so far.  Give a list of the currently suggested
answers. Everytime you propose an aswer, check this list.  You proposal cannot
be in this this list. Try again and submit a new unique answer."


In [14]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [15]:
from reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)



=== Attempt 1 ===
Let the hexagon A B C D E F be cyclic with center O and circumradius R.

- The central angles subtending the sides satisfy α = ∠AOB, β = ∠BOC, γ = ∠COD, and since AB = DE, BC = EF, CD = FA, we have α = ∠AOB = ∠DOE, β = ∠BOC = ∠EOF, γ = ∠COD = ∠FOA. Hence the six central angles are α, β, γ, α, β, γ and α + β + γ = 2π/2 = π, i.e. α + β + γ = π.

- Let x = α/2, y = β/2, z = γ/2. Then x, y, z > 0 and x + y + z = π/2. The given side lengths give
  AB = 2R sin x = √5,
  BC = 2R sin y = 3,
  CD = 2R sin z = 2√5.

  Thus sin x = √5/(2R), sin y = 3/(2R), sin z = √5/R.

- Since x + y + z = π/2, we have sin(x + y) = cos z. Also
  sin(x + y) = sin x cos y + cos x sin y,
  with cos y = √(1 − sin^2 y) and cos x = √(1 − sin^2 x), cos z = √(1 − sin^2 z).

  Substituting sin x, sin y, sin z gives the equation
  (√5/(2R)) √(1 − 9/(4R^2)) + (3/(2R)) √(1 − 5/(4R^2)) = √(1 − 5/R^2).

- Let t = R^2. After clearing radicals and simplifying (a routine but lengthy algebraic elimination), one

This solved the problem in 1 attempt.

### Tree of thought attempt

In [17]:
from dotenv import load_dotenv
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

load_dotenv()
config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=3,
    n_select_sample=3,
    n_generate_sample=3,
    steps=3,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ('Answer: R = sqrt((16 + √31)/2)\n\nReasoning (sketch of steps):\n- Let α, β, γ be the central angles for AB, BC, CD, with AB=DE, BC=EF, CD=FA, so α=δ, β=ε, γ=ζ and α+β+γ=π.\n- Set a=α/2, b=β/2, c=γ/2. Then a+b+c=π/2.\n- From chord lengths: sin a = AB/(2R) = √5/(2R), sin b = BC/(2R) = 3/(2R), sin c = CD/(2R) = 2√5/(2R) = √5/R.\n- Since a+b = π/2 − c, cos c = sin(a+b) = sin a cos b + cos a sin b.\n- Compute cos a = √(1 − sin^2 a) = √(4R^2 − 5)/(2R), cos b = √(4R^2 − 9)/(2R), cos c = √(R^2 − 5)/R.\n- This yields the equation √5√(4R^2−9) + 3√(4R^2−5) = 4R√(R^2−5).\n- Put t = R^2 and solve, after squaring twice, to get (t − 1)(4t^2 − 64t + 225) = 0.\n- The admissible root with R^2 > 5 is t = 8 + (√31)/2.\n- Therefore R^2 = 8 + (√31)/2 and R = sqrt(8 + (√31)/2) = sqrt((16 + √31)/2). (Numerical value ≈ 3.283.)', '- Let α, β, γ be the 

Model's final answer was sqrt((16 + √31)/2).

So, Tree of Thought got the right answer

In [ ]:
import numpy as np
(1+np.sqrt(31))/2

np.float64(3.2838821814150108)

In [20]:
np.sqrt((16+np.sqrt(31))/2)

np.float64(3.283882181415011)

### Solver-Rejector method - The proposed method

In [18]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem2" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

In this case, our method ranked the correct answer as the highest rank. Hence, our method worked best in this attempt among the two problems.

## Problem 3 - Difficulty Level 4

In [20]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==4].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                              MathArena/cmimc_2025: 5
answer                  ### Final answer\n\nReasoning:\n- The positive...
gold_answer                                                        194832
ten_percentile_group                                                    4
problem                 Consider all positive multiples of $77$ less t...
Name: 20160, dtype: object

In [21]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

Consider all positive multiples of $77$ less than $1,000,000$. What is the sum
of all the odd digits that show up?
answer 194832


### Reflexion Attempt

In [22]:
from multi_agent.multi_agent import Problem
from prompt_template import Reflexion_Solver, Reflector
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> Consider all positive multiples of $77$ less than
$1,000,000$. What is the sum of all the odd digits that show up?.<<  "When you
are done, you should submidt your answer as: ANSWER: <your answer>.  No latex
formatting, just the raw number/numbers or strings at the very end.  Before you
start sharing your toughts, give a little summary of the conversation so far.
Give a list of the currently suggested answers. Everytime you propose an aswer,
check this list.  You proposal cannot be in this this list. Try again and submit
a new unique answer."


In [23]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [24]:
from reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
Step-by-step solution:

1) Let the multiples be m_k = 77k for k = 1, 2, ..., 12,987 (since 77 × 12,987 = 999,999).

2) Look at the last digit of m_k. Since 77 ≡ 7 (mod 10), the last digit of m_k is the same as the last digit of 7k, i.e., m_k ≡ 7k (mod 10).

3) Because gcd(7, 10) = 1, the sequence 7k mod 10 (as k runs through any 10 consecutive integers) takes every residue 0 through 9 exactly once. In particular, among the first 10 multiples, the last digits are a permutation of 0,1,2,3,4,5,6,7,8,9.

4) Therefore, among all multiples m_k (k = 1 to 12,987), every digit 0–9 appears at least once as the last digit. In particular, the odd digits 1, 3, 5, 7, and 9 all appear somewhere in those numbers.

5) The problem asks for the sum of all the odd digits that show up (i.e., the distinct odd digits that appear). Those digits are {1, 3, 5, 7, 9}.

6) Sum = 1 + 3 + 5 + 7 + 9 = 25.

Answer: 25
Score: true
Feedback: Verdict: Correct.

Feedback: The solution correctly notes t

Here, in this case, Reflexion couldn't get to the correct answer!

### Tree of thought attempt

In [25]:
from dotenv import load_dotenv
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

load_dotenv()
config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=3,
    n_select_sample=3,
    n_generate_sample=3,
    steps=3,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

path /Users/nishadjahan/Documents/Scholarship/Denmark/Study/NLP/project/NLP-group-15
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ('Step: The multiples are 77n with 1 ≤ n ≤ floor(999999/77) = 12987, since 12987·77 = 999,999 < 1,000,000.\n\nNext steps:\n- The number 999,999 appears (n = 12,987), so the digit 9 appears.\n- The number 77 appears (n = 1), so the digit 7 appears.\n- The number 154 appears (n = 2), so digits 1 and 5 appear.\n- The number 231 appears (n = 3), so digit 3 appears.\n\nThus all odd digits {1, 3, 5, 7, 9} occur among these multiples.\n\nSum of all odd digits that show up = 1 + 3 + 5 + 7 + 9 = 25.\n\nAnswer: 25', 'Answer: 25', 'Answer: 25')
-- sol values --: (3, 0, 0)
-- choices --: ['Step: The multiples are 77n with 1 ≤ n ≤ floor(999999/77) = 12987, since 12987·77 = 999,999 < 1,000,000.\n\nNext steps:\n- The number 999,999 appears (n = 12,987), so the

In this case, Tree of Thought did not get the right answer.

### Solver-Rejector method - The proposed method

In [26]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem2" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

Our method got the answer in the second attempt but ranked it 2nd. Only our method ranked close to the right answer 194832 while the other methods didn't.

## Problem 4 - Difficulty Level 2

In [27]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==2].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                          MathArena/hmmt_feb_2025: 30
answer                  Let the six edges of the hexagonal cross-secti...
gold_answer                                          \sqrt{\frac{95}{24}}
ten_percentile_group                                                    2
problem                 A plane $\mathcal{P}$ intersects a rectangular...
Name: 15668, dtype: object

In [28]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

A plane $\mathcal{P}$ intersects a rectangular prism at a hexagon which has side
lengths $45,66,63,55,54$, and 77, in that order. Compute the distance from the
center of the rectangular prism to $\mathcal{P}$.
answer \sqrt{\frac{95}{24}}


### Reflexion Attempt

In [29]:
from multi_agent.multi_agent import Problem
from prompt_template import Reflexion_Solver, Reflector
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> A plane $\mathcal{P}$ intersects a rectangular prism at a
hexagon which has side lengths $45,66,63,55,54$, and 77, in that order. Compute
the distance from the center of the rectangular prism to $\mathcal{P}$..<<
"When you are done, you should submidt your answer as: ANSWER: <your answer>.
No latex formatting, just the raw number/numbers or strings at the very end.
Before you start sharing your toughts, give a little summary of the conversation
so far.  Give a list of the currently suggested answers. Everytime you propose
an aswer, check this list.  You proposal cannot be in this this list. Try again
and submit a new unique answer."


In [30]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [31]:
from reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
I can outline a full solution, but as stated the problem is underdetermined. To compute the distance from the center of the rectangular prism to the plane, we need more data than just the six side lengths of the cross-section hexagon. Specifically, we need the dimensions (edge lengths) of the prism itself (the three edge lengths of the box), or an explicit specification of how the six hexagon sides correspond to the six faces of the prism.

Why more data is needed (sketch of the idea):
- Let the prism be axis-aligned with center at the origin, with half-edge lengths a, b, c along the x-, y-, and z-axes respectively. The prism is then [-a,a] × [-b,b] × [-c,c].
- A plane P has equation αx + βy + γz = δ. Its distance to the center is d = |δ| / sqrt(α^2 + β^2 + γ^2).
- The intersection P ∩ prism is a hexagon whose six sides lie on the six faces x = ±a, y = ±b, z = ±c.
- Each edge lies on a fixed face and is the intersection of that face with the plane, i.e., on x = a we 

Reflexion is unable to answer this question.

### Tree of thought attempt

In [32]:
from dotenv import load_dotenv
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

load_dotenv()
config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=3,
    n_select_sample=3,
    n_generate_sample=3,
    steps=3,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ('Answer: \\sqrt{110}\n\nReasoning (sketch):\n- Let the prism be [-a,a]×[-b,b]×[-c,c], and the plane P have unit normal n = (n_x, n_y, n_z) and distance d from the center, so its equation is n_x x + n_y y + n_z z = d.\n- The hexagonal cross-section intersects the six faces; denote the lengths on opposite faces along x by s1 and s4, along y by s2 and s5, and along z by s3 and s6. The problem gives the six side lengths in order, so the three differences between opposite sides are\n  Δ1 = |s4 − s1|, Δ2 = |s5 − s2|, Δ3 = |s6 − s3|.\n- A standard property of such a cross-section is that the difference between the lengths on opposite faces is proportional to the plane’s offset from the center along the respective axis: Δ1 = 2 d |n_x|, Δ2 = 2 d |n_y|, Δ3 = 2 d |n_z|.\n- Squaring and summing gives Δ1^2 + Δ2^2 + Δ3^2 = 4 d^2 (n_x^2 + n_y

The Tree of Thought method proposed the answer sqrt(110). But the right answer is sqrt(95/24). So, it didn't give the right answer.

### Solver-Rejector method - The proposed method

In [33]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem2" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

In [34]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem2" ,n_steps=20, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

## Problem 5 - Difficulty Level 1

In [ ]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==1].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

In [ ]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

### Reflexion Attempt

In [ ]:
from multi_agent.multi_agent import Problem
from prompt_template import Reflexion_Solver, Reflector
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

In [ ]:
# Getting the GPT-5-nano model

from models.azure_api import Client 
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [ ]:
from reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)

### Tree of thought attempt

In [ ]:
from dotenv import load_dotenv
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

load_dotenv()
config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=3,
    n_select_sample=3,
    n_generate_sample=3,
    steps=3,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

### Solver-Rejector method - The proposed method

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="GPT-5_nano_chat_Problem2" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)